# Pharmaceutical Product Matching — Data Cleaning & Normalization

**Goal:** prepare pharmaceutical product data for **product matching, equivalent-product identification, and price comparison**.

This notebook is **not an ML notebook**. It does not train a model, scale features, or create train/test sets.

### Principles
- Preserve all original columns.
- Create cleaned/engineered columns separately.
- Never translate Arabic drug names automatically.
- Use deterministic rules first.
- Use fuzzy matching only as a candidate/review step.
- Do not consider products equivalent based on active ingredient alone.


In [2]:
# Imports
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

INPUT = Path("eg_drugs.csv")
OUTPUT = Path("eg_drugs_cleaned_product_matching.csv")

df = pd.read_csv(INPUT, dtype={"barcode": "string"})
work = df.copy()

print("Rows:", len(work))
print("Columns:", len(work.columns))
display(work.head())

Rows: 26562
Columns: 25


,id,name,arabic,active,company,price,oldprice,availability,barcode,slug,units,description,uses,matched_fda_ingredients,uses_summary,uses_summary_en,warning_high_blood_pressure,warning_diabetes,warning_pregnancy,warning_lactation,warning_kidney,warning_liver,warning_heart,warnings_summary,warnings_summary_en
0,1,1 2 3 (one two three) syrup 120 ml,ون تو ثري شراب 120 مل,Chlorpheniramine+paracetamol+pseudoephedrine,Hikma,40.00.,32.00.,NaN,6221000000010,1,1,Cold drugs,تخفيف أعراض نزلات البرد والإنفلونزا مثل الصداع وارتفاع درجة الحرارة. يساعد في علاج احتقان الأنف والجيوب الأنفية وتحس...,pseudoephedrine | chlorpheniramine,تخفيف أعراض نزلات البرد والإنفلونزا مثل الصداع وارتفاع درجة الحرارة. يساعد في علاج احتقان الأنف والجيوب الأنفية وتحس...,"temporarily relieves nasal congestion due to the common cold, hay fever or other upper respiratory allergies tempora...",1,1,1,1,0,0,1,يحذر أو يوصى بالحذر تحت إشراف طبي في الحالات التالية: الحمل، الرضاعة الطبيعية، ارتفاع ضغط الدم، مرض السكري، أمراض ال...,"Caution or warning advised under medical supervision for: Pregnancy, Lactation/Breastfeeding, High Blood Pressure, D..."
1,2,1 2 3 (one two three) 20 f.c.tabs,ون تو ثري 20 قرص,Chlorpheniramine+paracetamol+pseudoephedrine,Hikma,40.00.,24.00.,NaN,6221000000010,2,2,Cold drugs,تخفيف أعراض نزلات البرد والإنفلونزا مثل الصداع وارتفاع درجة الحرارة. يساعد في علاج احتقان الأنف والجيوب الأنفية وتحس...,pseudoephedrine | chlorpheniramine,تخفيف أعراض نزلات البرد والإنفلونزا مثل الصداع وارتفاع درجة الحرارة. يساعد في علاج احتقان الأنف والجيوب الأنفية وتحس...,"temporarily relieves nasal congestion due to the common cold, hay fever or other upper respiratory allergies tempora...",1,1,1,1,0,0,1,يحذر أو يوصى بالحذر تحت إشراف طبي في الحالات التالية: الحمل، الرضاعة الطبيعية، ارتفاع ضغط الدم، مرض السكري، أمراض ال...,"Caution or warning advised under medical supervision for: Pregnancy, Lactation/Breastfeeding, High Blood Pressure, D..."
2,3,5-fluorouracil-ebewe 250mg/5ml i.v. vial,فلورويوراسيل 250مجم/5مل فيال وريد,Fluorouracil,Shanghai xudong haipu pharmaceutical > acdima international trading,74.00.,50.00.,NaN,9088881320685,3,1,Antineoplastic,يُستخدم في علاج بعض أنواع سرطان القولون والمستقيم ويُعد من الأدوية الأساسية في بروتوكولات العلاج الكيماوي. يُستعمل ف...,fluorouracil,يُستخدم في علاج بعض أنواع سرطان القولون والمستقيم ويُعد من الأدوية الأساسية في بروتوكولات العلاج الكيماوي. يُستعمل ف...,Fluorouracil Cream is recommended for the topical treatment of multiple actinic or solar keratoses.,0,0,1,0,0,0,0,يحذر أو يوصى بالحذر تحت إشراف طبي في الحالات التالية: الحمل.,Caution or warning advised under medical supervision for: Pregnancy.
3,4,mooagar sachets,مو اجار 2جم اكياس,agar,hexa pharma company,75,NaN,3710.0,<NA>,mooagar-sachets,1,jaundice,الصفراء عند حديثي الولادة يُستخدم كمُليّن طبيعي لعلاج الإمساك، حيث يساعد على زيادة حجم البراز وتسهيل خروجه. يُستعمل ...,NaN,الصفراء عند حديثي الولادة يُستخدم كمُليّن طبيعي لعلاج الإمساك، حيث يساعد على زيادة حجم البراز وتسهيل خروجه. يُستعمل ...,No registered indications.,0,0,0,0,0,0,0,لا توجد تحذيرات خاصة مسجلة لهذه الحالات. استشر الطبيب دائمًا.,No special warnings registered for these conditions. Always consult a doctor.
4,5,limitless lactoferrin 100 mg 30 sachets,ليمتلس لاكتوفيرين 100 مجم 30 كيس,lactoferrin,eva pharma,485,385,3711.0,<NA>,limitless-lactoferrin-100-mg-30-sachets,1,iron supplement,يساعد في علاج الأنيميا الناتجة عن نقص الحديد لأنه يحسن امتصاص الحديد في الجسم. يساهم في تقوية جهاز المناعة ومقاومة ا...,NaN,يساعد في علاج الأنيميا الناتجة عن نقص الحديد لأنه يحسن امتصاص الحديد في الجسم. يساهم في تقوية جهاز المناعة ومقاومة ا...,No registered indications.,0,0,0,0,0,0,0,لا توجد تحذيرات خاصة مسجلة لهذه الحالات. استشر الطبيب دائمًا.,No special warnings registered for these conditions. Always consult a doctor.


## 1. Initial Data Quality Check

In [3]:
print("Shape:", work.shape)
print("\nData types:")
display(work.dtypes.to_frame("dtype"))

quality = pd.DataFrame({
    "dtype": work.dtypes.astype(str),
    "missing_count": work.isna().sum(),
    "missing_pct": (work.isna().mean() * 100).round(2),
    "unique_values": work.nunique(dropna=True)
}).sort_values("missing_count", ascending=False)

display(quality)
print("Exact duplicate rows:", work.duplicated().sum())

Shape: (26562, 25)

Data types:


,dtype
id,int64
name,str
arabic,str
active,str
company,str
price,str
oldprice,str
availability,float64
barcode,string
slug,str


,dtype,missing_count,missing_pct,unique_values
uses,str,18455,69.48,5111
barcode,string,16030,60.35,10139
oldprice,str,15200,57.22,1383
matched_fda_ingredients,str,6980,26.28,5887
active,str,2757,10.38,9102
description,str,1135,4.27,2909
availability,float64,343,1.29,208
company,str,310,1.17,5533
price,str,5,0.02,2319
slug,str,1,0.00,26560


Exact duplicate rows: 0


## 2. General Text Normalization

Arabic is preserved. Only conservative Unicode/whitespace normalization and selected Arabic variants are applied for matching-oriented fields.

In [4]:
def normalize_text(value, normalize_arabic=False, normalize_separators=False):
    if pd.isna(value):
        return pd.NA

    s = unicodedata.normalize("NFKC", str(value))
    s = s.replace("\u200f", "").replace("\u200e", "").replace("\ufeff", "")

    if normalize_arabic:
        # Conservative matching normalization; original value is untouched.
        s = s.translate(str.maketrans({
            "أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا", "ى": "ي"
        }))

    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()

    if normalize_separators:
        s = re.sub(r"\s*([,+|>/])\s*", r"\1", s)

    return s

text_rules = {
    "name": ("name_clean", True, True),
    "arabic": ("arabic_clean", True, False),
    "active": ("active_clean", False, True),
    "company": ("company_clean", True, True),
}

for src, (dst, arabic, separators) in text_rules.items():
    if src in work.columns:
        work[dst] = work[src].map(lambda x: normalize_text(x, arabic, separators))

display(work[[c for c in ["name", "name_clean", "arabic", "arabic_clean", "active", "active_clean", "company", "company_clean"] if c in work.columns]].head(10))

,name,name_clean,arabic,arabic_clean,active,active_clean,company,company_clean
0,1 2 3 (one two three) syrup 120 ml,1 2 3 (one two three) syrup 120 ml,ون تو ثري شراب 120 مل,ون تو ثري شراب 120 مل,Chlorpheniramine+paracetamol+pseudoephedrine,chlorpheniramine+paracetamol+pseudoephedrine,Hikma,hikma
1,1 2 3 (one two three) 20 f.c.tabs,1 2 3 (one two three) 20 f.c.tabs,ون تو ثري 20 قرص,ون تو ثري 20 قرص,Chlorpheniramine+paracetamol+pseudoephedrine,chlorpheniramine+paracetamol+pseudoephedrine,Hikma,hikma
2,5-fluorouracil-ebewe 250mg/5ml i.v. vial,5-fluorouracil-ebewe 250mg/5ml i.v. vial,فلورويوراسيل 250مجم/5مل فيال وريد,فلورويوراسيل 250مجم/5مل فيال وريد,Fluorouracil,fluorouracil,Shanghai xudong haipu pharmaceutical > acdima international trading,shanghai xudong haipu pharmaceutical>acdima international trading
3,mooagar sachets,mooagar sachets,مو اجار 2جم اكياس,مو اجار 2جم اكياس,agar,agar,hexa pharma company,hexa pharma company
4,limitless lactoferrin 100 mg 30 sachets,limitless lactoferrin 100 mg 30 sachets,ليمتلس لاكتوفيرين 100 مجم 30 كيس,ليمتلس لاكتوفيرين 100 مجم 30 كيس,lactoferrin,lactoferrin,eva pharma,eva pharma
5,plovimas mouth spray 60 ml,plovimas mouth spray 60 ml,بلوفيماس سبراي للفم 60 مل,بلوفيماس سبراي للفم 60 مل,des-infectant formula,des-infectant formula,NaN,NaN
6,a1 cream 100 gm,a1 cream 100 gm,ايه وان بيبي كريم 100جم,ايه وان بيبي كريم 100جم,castor oil+zinc oxide+arachis oil+bees wax,castor oil+zinc oxide+arachis oil+bees wax,egpi > egd,egpi>egd
7,limitless caffeine 80mg 20 oral dispersible films,limitless caffeine 80mg 20 oral dispersible films,ليميتلس كافيين 80مجم 20 فيلم,ليميتلس كافيين 80مجم 20 فيلم,caffeine,caffeine,eve pharma,eve pharma
8,no cal 300 packets,no cal 300 packets,نو كال 300 باكت,نو كال 300 باكت,sucralose+dextrose+maltodextrin,sucralose+dextrose+maltodextrin,NaN,NaN
9,canti colin syrup 100 ml,canti colin syrup 100 ml,كانتي كولين شراب 100 مل,كانتي كولين شراب 100 مل,citicoline,citicoline,pharma zad > cantreat,pharma zad>cantreat


## 3. Numeric / Barcode Cleaning

In [5]:
def clean_money(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().replace(",", "")
    s = re.sub(r"\.+$", "", s)
    if re.fullmatch(r"[+-]?\d+(?:\.\d+)?", s):
        return float(s)
    return np.nan

def clean_barcode(value):
    if pd.isna(value):
        return pd.NA
    s = str(value).strip()
    # Convert Excel-like 123456789.0 without destroying leading zeros.
    if re.fullmatch(r"\d+\.0+", s):
        s = s.split(".")[0]
    return s or pd.NA

if "price" in work.columns:
    work["price_clean"] = work["price"].map(clean_money)
if "oldprice" in work.columns:
    work["oldprice_clean"] = work["oldprice"].map(clean_money)
if "availability" in work.columns:
    raw_availability = pd.to_numeric(work["availability"], errors="coerce")
    work["availability_clean"] = raw_availability.round().astype("Int64")
    work["negative_availability_flag"] = raw_availability.lt(0)
    work.loc[work["negative_availability_flag"], "availability_clean"] = pd.NA
if "barcode" in work.columns:
    work["barcode_clean"] = work["barcode"].map(clean_barcode).astype("string")
if "units" in work.columns:
    work["units_clean"] = pd.to_numeric(work["units"], errors="coerce").round().astype("Int64")

# Validation flags — do not silently fix questionable records.
if "price_clean" in work:
    work["price_invalid_flag"] = work["price"].notna() & work["price_clean"].isna()
if "oldprice_clean" in work:
    work["oldprice_invalid_flag"] = work["oldprice"].notna() & work["oldprice_clean"].isna()
if {"price_clean", "oldprice_clean"}.issubset(work.columns):
    work["oldprice_less_than_current_flag"] = (
        work["oldprice_clean"].notna() & work["price_clean"].notna() &
        work["oldprice_clean"].lt(work["price_clean"])
    )

if "barcode_clean" in work:
    work["barcode_format_flag"] = work["barcode_clean"].notna() & ~work["barcode_clean"].str.fullmatch(r"\d{8,14}", na=False)
    barcode_counts = work.loc[work["barcode_clean"].notna(), "barcode_clean"].value_counts()
    work["barcode_duplicate_flag"] = work["barcode_clean"].map(barcode_counts).fillna(0).gt(1)

display(work[[c for c in ["price", "price_clean", "oldprice", "oldprice_clean", "availability", "availability_clean", "barcode", "barcode_clean"] if c in work.columns]].head(10))

,price,price_clean,oldprice,oldprice_clean,availability,availability_clean,barcode,barcode_clean
0,40.00.,40.0,32.00.,32.0,NaN,<NA>,6221000000010,6221000000010
1,40.00.,40.0,24.00.,24.0,NaN,<NA>,6221000000010,6221000000010
2,74.00.,74.0,50.00.,50.0,NaN,<NA>,9088881320685,9088881320685
3,75,75.0,NaN,NaN,3710.0,3710,<NA>,<NA>
4,485,485.0,385,385.0,3711.0,3711,<NA>,<NA>
5,79,79.0,NaN,NaN,3621.0,3621,<NA>,<NA>
6,45,45.0,NaN,NaN,3837.0,3837,6223003930479,6223003930479
7,145,145.0,128,128.0,3496.0,3496,6223012483706,6223012483706
8,340,340.0,NaN,NaN,3789.0,3789,6224000843083,6224000843083
9,200,200.0,NaN,NaN,784.0,784,721688507246,721688507246


## 4. Active Ingredient Canonicalization

In [6]:
def canonical_active(value):
    if pd.isna(value) or not str(value).strip():
        return pd.NA

    parts = [p.strip() for p in re.split(r"\s*[+|,]\s*", str(value)) if p.strip()]
    # Sort only the ingredient tokens; this creates a canonical representation
    # for combinations where ingredient order does not define identity.
    return "+".join(sorted(dict.fromkeys(parts)))

if "active_clean" in work:
    work["active_canonical"] = work["active_clean"].map(canonical_active)

if "active_clean" in work:
    work["missing_active_flag"] = work["active_clean"].isna() | work["active_clean"].eq("")


## 5. Strength / Concentration / Volume Extraction

The parser distinguishes simple strength from concentration such as `5 mg/5 ml`. Multi-strength expressions such as `10/5/1000 mg` are preserved as a structured string.

In [7]:
NUM = r"(?:\d+(?:\.\d+)?|\.\d+)"
UNIT = r"(?:mg|mcg|µg|ug|g|kg|ng|iu|miu|mmol|mol|%|meq|kcal)"

concentration_pattern = re.compile(
    rf"({NUM})\s*({UNIT})\s*/\s*(?:(\d+(?:\.\d+)?)\s*)?(ml|l|g|kg|mg)\b", re.I
)
multi_strength_pattern = re.compile(rf"((?:{NUM}\s*/\s*)+{NUM})\s*({UNIT})\b", re.I)
single_strength_pattern = re.compile(rf"({NUM})\s*({UNIT})\b", re.I)

def normalize_unit(unit):
    return unit.lower().replace("µg", "mcg").replace("ug", "mcg")

def parse_dose(value):
    result = {
        "strength_value": np.nan,
        "strength_values": pd.NA,
        "strength_unit": pd.NA,
        "concentration_value": np.nan,
        "concentration_unit": pd.NA,
        "concentration_denominator_value": np.nan,
        "volume_value_from_concentration": np.nan,
        "volume_unit_from_concentration": pd.NA,
    }
    if pd.isna(value):
        return result

    s = str(value)
    m = concentration_pattern.search(s)
    if m:
        result.update({
            "concentration_value": float(m.group(1)),
            "concentration_unit": normalize_unit(m.group(2)),
            "concentration_denominator_value": float(m.group(3)) if m.group(3) else 1.0,
            "volume_value_from_concentration": float(m.group(3)) if m.group(3) else 1.0,
            "volume_unit_from_concentration": m.group(4).lower(),
        })
        return result

    m = multi_strength_pattern.search(s)
    if m:
        values = [float(x) for x in m.group(1).split("/")]
        normalized = "/".join(str(int(x)) if x.is_integer() else str(x) for x in values)
        result.update({
            "strength_value": values[0],
            "strength_values": normalized,
            "strength_unit": normalize_unit(m.group(2)),
        })
        return result

    m = single_strength_pattern.search(s)
    if m:
        value = float(m.group(1))
        result.update({
            "strength_value": value,
            "strength_values": str(int(value)) if value.is_integer() else str(value),
            "strength_unit": normalize_unit(m.group(2)),
        })
    return result

if "name_clean" in work:
    dose = pd.DataFrame(work["name_clean"].map(parse_dose).tolist(), index=work.index)
    for column in dose.columns:
        work[column] = dose[column]


## 6. Dosage Form, Route, Release Type, Pack Size and Volume

In [8]:
FORM_PATTERNS = [
    ("oral dispersible tablet", r"\b(?:oral\s+dispersible|odt)\s+(?:tablet|tablets|tab|tabs)\b"),
    ("tablet", r"\b(?:f\.?c\.?\s*)?(?:tablets?|tabs?)\b"),
    ("capsule", r"\b(?:capsules?|caps?)\b"),
    ("syrup", r"\bsyrups?\b"),
    ("suspension", r"\bsuspensions?\b"),
    ("solution", r"\bsolutions?\b"),
    ("injection", r"\b(?:injections?|inj)\b"),
    ("ampoule", r"\b(?:ampoules?|amps?|amp)\b"),
    ("vial", r"\bvials?\b"),
    ("cream", r"\bcreams?\b"),
    ("ointment", r"\bointments?\b"),
    ("gel", r"\bgels?\b"),
    ("drops", r"\bdrops?\b"),
    ("spray", r"\bsprays?\b"),
    ("suppository", r"\bsuppositories?\b"),
    ("sachet", r"\bsachets?\b"),
    ("oral film", r"\boral\s+(?:dispersible\s+)?films?\b"),
    ("powder", r"\bpowders?\b"),
    ("lozenge", r"\blozenges?\b"),
    ("patch", r"\bpatches?\b"),
    ("lotion", r"\blotions?\b"),
    ("shampoo", r"\bshampoos?\b"),
    ("mouthwash", r"\bmouthwash(?:es)?\b"),
]

def detect_form(value):
    if pd.isna(value):
        return "unknown"
    s = str(value)
    for form_name, pattern in FORM_PATTERNS:
        if re.search(pattern, s):
            return form_name
    return "unknown"

if "name_clean" in work:
    work["dosage_form"] = work["name_clean"].map(detect_form)

ROUTE_BY_FORM = {
    "tablet": "oral", "oral dispersible tablet": "oral", "capsule": "oral",
    "syrup": "oral", "suspension": "oral", "sachet": "oral", "oral film": "oral",
    "lozenge": "oral", "powder": "oral", "cream": "topical", "ointment": "topical",
    "gel": "topical", "lotion": "topical", "shampoo": "topical", "patch": "topical",
    "vial": "injectable", "ampoule": "injectable", "injection": "injectable",
}
work["route"] = work["dosage_form"].map(ROUTE_BY_FORM).fillna("unknown")

RELEASE_PATTERNS = [
    ("extended release", r"\b(?:extended|sustained|prolonged|controlled|modified)\s+release\b|\b(?:xr|er|sr|cr|mr)\b"),
    ("delayed release", r"\bdelayed\s+release\b|\bdr\b"),
    ("enteric coated", r"\benteric(?:-|\s+)coated\b|\be\.?c\.?\b"),
    ("immediate release", r"\bimmediate\s+release\b"),
]

def detect_release(value):
    if pd.isna(value):
        return "unknown"
    for release_name, pattern in RELEASE_PATTERNS:
        if re.search(pattern, str(value)):
            return release_name
    return "unknown"

work["release_type"] = work["name_clean"].map(detect_release)

FORM_WORDS = r"(?:tabs?|tablets?|caps?|capsules?|sachets?|vials?|amp(?:oules?)?|suppositories?|films?|lozenges?|patches?|bottles?)"

def extract_pack_size(value):
    if pd.isna(value):
        return np.nan
    s = str(value)
    patterns = [
        rf"\b(\d+)\s*(?:f\.?c\.?\s*)?{FORM_WORDS}\b",
        rf"\b{FORM_WORDS}\s*(?:x\s*)?(\d+)\b",
        r"\b(?:pack|box)\s*(?:of\s*)?(\d+)\b",
    ]
    for pattern in patterns:
        match = re.search(pattern, s)
        if match:
            return int(match.group(1))
    return np.nan

def extract_volume(value):
    if pd.isna(value):
        return np.nan, pd.NA
    matches = list(re.finditer(rf"\b({NUM})\s*(ml|l)\b", str(value), re.I))
    if not matches:
        return np.nan, pd.NA
    match = matches[-1]
    return float(match.group(1)), match.group(2).lower()

work["pack_size"] = work["name_clean"].map(extract_pack_size)
volume_df = pd.DataFrame(work["name_clean"].map(extract_volume).tolist(), columns=["volume_value", "volume_unit"], index=work.index)
work[["volume_value", "volume_unit"]] = volume_df


## 7. Product Matching Keys

Two levels are intentionally kept:
- **product_family_key:** same active/strength/form/route/release family, packaging may differ.
- **product_match_key:** stricter key for direct product/price comparison; packaging or volume is included.

These keys are deterministic and are not ML features.

In [9]:
LIQUID_FORMS = {"syrup", "suspension", "solution", "drops", "injection", "vial", "ampoule", "spray", "mouthwash"}

def dose_key(row):
    if pd.notna(row.get("concentration_value")):
        return (
            f"conc:{row['concentration_value']:g}:"
            f"{row['concentration_unit']}/"
            f"{row['concentration_denominator_value']:g}:"
            f"{row['volume_unit_from_concentration']}"
        )
    if pd.notna(row.get("strength_values")):
        return f"strength:{row['strength_values']}:{row['strength_unit']}"
    return None

def build_family_key(row):
    active = row.get("active_canonical")
    form = row.get("dosage_form")
    if pd.isna(active) or form == "unknown":
        return pd.NA
    dose = dose_key(row)
    if dose is None:
        return pd.NA
    return "|".join([str(active), dose, form, str(row.get("route", "unknown")), str(row.get("release_type", "unknown"))])

work["product_family_key"] = work.apply(build_family_key, axis=1)

def build_product_match_key(row):
    family = row.get("product_family_key")
    if pd.isna(family):
        return pd.NA

    form = row.get("dosage_form")
    if form in LIQUID_FORMS:
        if pd.isna(row.get("volume_value")) or pd.isna(row.get("volume_unit")):
            return pd.NA
        return f"{family}|volume:{row['volume_value']:g}:{row['volume_unit']}"

    if pd.isna(row.get("pack_size")):
        return pd.NA
    return f"{family}|pack:{int(row['pack_size'])}"

work["product_match_key"] = work.apply(build_product_match_key, axis=1)

print("Strict matching keys:", work["product_match_key"].notna().sum())
print("Family keys:", work["product_family_key"].notna().sum())

Strict matching keys: 7984
Family keys: 9795


## 8. Duplicate / Candidate Match Reports

In [10]:
# Exact duplicate rows
exact_duplicates = work[work.duplicated(keep=False)].copy()

# Same strict product key = strong candidate for same product/pack
product_duplicates = (
    work[work["product_match_key"].notna()]
    .loc[lambda x: x["product_match_key"].duplicated(keep=False)]
    .sort_values("product_match_key")
)

# Same family key = same normalized product family, but packaging may differ
family_duplicates = (
    work[work["product_family_key"].notna()]
    .loc[lambda x: x["product_family_key"].duplicated(keep=False)]
    .sort_values("product_family_key")
)

# Barcode duplicates
barcode_duplicates = work[work.get("barcode_duplicate_flag", False)].sort_values("barcode_clean") if "barcode_duplicate_flag" in work else pd.DataFrame()

print("Exact duplicate rows:", len(exact_duplicates))
print("Rows in repeated strict product keys:", len(product_duplicates))
print("Rows in repeated family keys:", len(family_duplicates))
print("Rows with duplicate barcodes:", len(barcode_duplicates))

display(product_duplicates.head(20))

Exact duplicate rows: 0
Rows in repeated strict product keys: 4796
Rows in repeated family keys: 7601
Rows with duplicate barcodes: 772


,id,name,arabic,active,company,price,oldprice,availability,barcode,slug,units,description,uses,matched_fda_ingredients,uses_summary,uses_summary_en,warning_high_blood_pressure,warning_diabetes,warning_pregnancy,warning_lactation,warning_kidney,warning_liver,warning_heart,warnings_summary,warnings_summary_en,name_clean,arabic_clean,active_clean,company_clean,price_clean,oldprice_clean,availability_clean,negative_availability_flag,barcode_clean,units_clean,price_invalid_flag,oldprice_invalid_flag,oldprice_less_than_current_flag,barcode_format_flag,barcode_duplicate_flag,active_canonical,missing_active_flag,strength_value,strength_values,strength_unit,concentration_value,concentration_unit,concentration_denominator_value,volume_value_from_concentration,volume_unit_from_concentration,dosage_form,route,release_type,pack_size,volume_value,volume_unit,product_family_key,product_match_key
23188,30066,zytiga 250 mg 120 tabs,زيتيجا 250مجم 120 اقراص,abiraterone acetate,janssen cilag > soficopharm,21685,NaN,0.0,<NA>,zytiga-250-mg-120-tabs,12,steroidal antiandrogen,NaN,abiraterone acetate,دواعي الاستعمال: 1 INDICATIONS AND USAGE Abiraterone acetate tablets are indicated in combination with prednisone fo...,1 INDICATIONS AND USAGE Abiraterone acetate tablets are indicated in combination with prednisone for the treatment o...,1,1,0,0,0,0,1,يحذر أو يوصى بالحذر تحت إشراف طبي في الحالات التالية: ارتفاع ضغط الدم، مرض السكري، أمراض القلب.,"Caution or warning advised under medical supervision for: High Blood Pressure, Diabetes, Heart Disease.",zytiga 250 mg 120 tabs,زيتيجا 250مجم 120 اقراص,abiraterone acetate,janssen cilag>soficopharm,21685.0,NaN,0,False,<NA>,12,False,False,False,False,False,abiraterone acetate,False,250.0,250,mg,NaN,NaN,NaN,NaN,NaN,tablet,oral,unknown,120.0,NaN,NaN,abiraterone acetate|strength:250:mg|tablet|oral|unknown,abiraterone acetate|strength:250:mg|tablet|oral|unknown|pack:120
23938,30862,mantroxate 250 mg 120 tabs,مانتروكسات 250مجم 120 قرص,abiraterone acetate,hikma specialized pharmaceuticals,26600,12853,0.0,<NA>,mantroxate-250-mg-120-tabs,1,steroidal antiandrogen,يعالج أبيراتيرون سرطان البروستاتا. ويعمل عن طريق خفض مستويات هرمون التستوستيرون في الجسم. وهذا يمنع خلايا سرطان البر...,abiraterone acetate,يعالج أبيراتيرون سرطان البروستاتا. ويعمل عن طريق خفض مستويات هرمون التستوستيرون في الجسم.,1 INDICATIONS AND USAGE Abiraterone acetate tablets are indicated in combination with prednisone for the treatment o...,1,1,0,0,0,0,1,يحذر أو يوصى بالحذر تحت إشراف طبي في الحالات التالية: ارتفاع ضغط الدم، مرض السكري، أمراض القلب.,"Caution or warning advised under medical supervision for: High Blood Pressure, Diabetes, Heart Disease.",mantroxate 250 mg 120 tabs,مانتروكسات 250مجم 120 قرص,abiraterone acetate,hikma specialized pharmaceuticals,26600.0,12853.0,0,False,<NA>,1,False,False,True,False,False,abiraterone acetate,False,250.0,250,mg,NaN,NaN,NaN,NaN,NaN,tablet,oral,unknown,120.0,NaN,NaN,abiraterone acetate|strength:250:mg|tablet|oral|unknown,abiraterone acetate|strength:250:mg|tablet|oral|unknown|pack:120
9649,13310,aceclocopa 100 mg 20 f.c. tab,اسيكلوكوبا 100 مجم 20 قرص,aceclofenac,copad pharma,22.5,NaN,0.0,6224000394226,aceclocopa-100-mg-20-f.c.-tab,2,nsaid.acetic acid derivatives,مسكن للالام -- خافض للحرارة -- تقليل الالتهابات والالام,NaN,مسكن للالام -- خافض للحرارة -- تقليل الالتهابات والالام.,No registered indications.,0,0,0,0,0,0,0,لا توجد تحذيرات خاصة مسجلة لهذه الحالات. استشر الطبيب دائمًا.,No special warnings registered for these conditions. Always consult a doctor.,aceclocopa 100 mg 20 f.c. tab,اسيكلوكوبا 100 مجم 20 قرص,aceclofenac,copad pharma,22.5,NaN,0,False,6224000394226,2,False,False,False,False,False,aceclofenac,False,100.0,100,mg,NaN,NaN,NaN,NaN,NaN,tablet,oral,unknown,20.0,NaN,NaN,aceclofenac|strength:100:mg|tablet|oral|unknown,aceclofenac|strength:100:mg|tablet|oral|unknown|pack:20
9650,13311,aceliofinaz 100 mg 20 f.c. tab,اسيليوفيناز 100 مجم 20 قرص,aceclofenac,aug > people pharma,56,39,0.0,6224008819066,aceliofinaz

## 9. Validation / Manual Review

Questionable values are flagged, not silently corrected.

In [11]:
review_flags = [c for c in [
    "price_invalid_flag", "oldprice_invalid_flag", "negative_availability_flag",
    "barcode_format_flag", "barcode_duplicate_flag", "oldprice_less_than_current_flag",
    "missing_name_flag", "missing_active_flag"
] if c in work.columns]

if "name_clean" in work:
    work["missing_name_flag"] = work["name_clean"].isna() | work["name_clean"].eq("")

review_mask = work[review_flags].any(axis=1) if review_flags else pd.Series(False, index=work.index)
validation_review = work.loc[review_mask].copy()

print("Records requiring review:", len(validation_review))
display(validation_review.head(20))

Records requiring review: 13360


,id,name,arabic,active,company,price,oldprice,availability,barcode,slug,units,description,uses,matched_fda_ingredients,uses_summary,uses_summary_en,warning_high_blood_pressure,warning_diabetes,warning_pregnancy,warning_lactation,warning_kidney,warning_liver,warning_heart,warnings_summary,warnings_summary_en,name_clean,arabic_clean,active_clean,company_clean,price_clean,oldprice_clean,availability_clean,negative_availability_flag,barcode_clean,units_clean,price_invalid_flag,oldprice_invalid_flag,oldprice_less_than_current_flag,barcode_format_flag,barcode_duplicate_flag,active_canonical,missing_active_flag,strength_value,strength_values,strength_unit,concentration_value,concentration_unit,concentration_denominator_value,volume_value_from_concentration,volume_unit_from_concentration,dosage_form,route,release_type,pack_size,volume_value,volume_unit,product_family_key,product_match_key,missing_name_flag
0,1,1 2 3 (one two three) syrup 120 ml,ون تو ثري شراب 120 مل,Chlorpheniramine+paracetamol+pseudoephedrine,Hikma,40.00.,32.00.,NaN,6221000000010,1,1,Cold drugs,تخفيف أعراض نزلات البرد والإنفلونزا مثل الصداع وارتفاع درجة الحرارة. يساعد في علاج احتقان الأنف والجيوب الأنفية وتحس...,pseudoephedrine | chlorpheniramine,تخفيف أعراض نزلات البرد والإنفلونزا مثل الصداع وارتفاع درجة الحرارة. يساعد في علاج احتقان الأنف والجيوب الأنفية وتحس...,"temporarily relieves nasal congestion due to the common cold, hay fever or other upper respiratory allergies tempora...",1,1,1,1,0,0,1,يحذر أو يوصى بالحذر تحت إشراف طبي في الحالات التالية: الحمل، الرضاعة الطبيعية، ارتفاع ضغط الدم، مرض السكري، أمراض ال...,"Caution or warning advised under medical supervision for: Pregnancy, Lactation/Breastfeeding, High Blood Pressure, D...",1 2 3 (one two three) syrup 120 ml,ون تو ثري شراب 120 مل,chlorpheniramine+paracetamol+pseudoephedrine,hikma,40.0,32.0,<NA>,False,6221000000010,1,False,False,True,False,True,chlorpheniramine+paracetamol+pseudoephedrine,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,syrup,oral,unknown,NaN,120.0,ml,NaN,NaN,False
1,2,1 2 3 (one two three) 20 f.c.tabs,ون تو ثري 20 قرص,Chlorpheniramine+paracetamol+pseudoephedrine,Hikma,40.00.,24.00.,NaN,6221000000010,2,2,Cold drugs,تخفيف أعراض نزلات البرد والإنفلونزا مثل الصداع وارتفاع درجة الحرارة. يساعد في علاج احتقان الأنف والجيوب الأنفية وتحس...,pseudoephedrine | chlorpheniramine,تخفيف أعراض نزلات البرد والإنفلونزا مثل الصداع وارتفاع درجة الحرارة. يساعد في علاج احتقان الأنف والجيوب الأنفية وتحس...,"temporarily relieves nasal congestion due to the common cold, hay fever or other upper respiratory allergies tempora...",1,1,1,1,0,0,1,يحذر أو يوصى بالحذر تحت إشراف طبي في الحالات التالية: الحمل، الرضاعة الطبيعية، ارتفاع ضغط الدم، مرض السكري، أمراض ال...,"Caution or warning advised under medical supervision for: Pregnancy, Lactation/Breastfeeding, High Blood Pressure, D...",1 2 3 (one two three) 20 f.c.tabs,ون تو ثري 20 قرص,chlorpheniramine+paracetamol+pseudoephedrine,hikma,40.0,24.0,<NA>,False,6221000000010,2,False,False,True,False,True,chlorpheniramine+paracetamol+pseudoephedrine,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,tablet,oral,unknown,20.0,NaN,NaN,NaN,NaN,False
2,3,5-fluorouracil-ebewe 250mg/5ml i.v. vial,فلورويوراسيل 250مجم/5مل فيال وريد,Fluorouracil,Shanghai xudong haipu pharmaceutical > acdima international trading,74.00.,50.00.,NaN,9088881320685,3,1,Antineoplastic,يُستخدم في علاج بعض أنواع سرطان القولون والمستقيم ويُعد من الأدوية الأساسية في بروتوكولات العلاج الكيماوي. يُستعمل ف...,fluorouracil,يُستخدم في علاج بعض أنواع سرطان القولون والمستقيم ويُعد من الأدوية الأساسية في بروتوكولات العلاج الكيماوي. يُستعمل ف...,Fluorouracil Cream is recommended for the topical treatment of multiple actinic or solar keratoses.,0,0,1,0,0,0,0,يحذر أو يوصى بالحذر تحت إشراف طبي في الحالات التالية: الحمل.,Caution or warning advised under medical supervision for: Pregnancy.,5-fluorouracil-ebewe 250mg/5ml i.v. vial,فلورويوراسيل 250مجم/5مل فيال وريد,fluorouracil,shanghai xudong haipu pharmaceutical>acdima international trading,74.0,50.0

## 10. Price Comparison Helper

Use `product_match_key` for direct price comparison. Availability is checked separately because missing availability must not automatically mean unavailable.

In [12]:
def cheapest_available_products(data, product_match_key, min_availability=1):
    required = {"product_match_key", "price_clean", "availability_clean"}
    missing = required - set(data.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    candidates = data[
        data["product_match_key"].eq(product_match_key) &
        data["price_clean"].notna() &
        data["availability_clean"].ge(min_availability)
    ].copy()

    return candidates.sort_values("price_clean")

# Example usage:
# key = work.loc[0, "product_match_key"]
# cheapest_available_products(work, key).head(10)

## 11. Export Clean Dataset and Reports

In [13]:
# Keep all original columns + cleaned/matching columns.
work.to_csv(OUTPUT, index=False, encoding="utf-8-sig")

exact_duplicates.to_csv("eg_drugs_exact_duplicates.csv", index=False, encoding="utf-8-sig")
product_duplicates.to_csv("eg_drugs_product_duplicates.csv", index=False, encoding="utf-8-sig")
family_duplicates.to_csv("eg_drugs_family_duplicates.csv", index=False, encoding="utf-8-sig")
barcode_duplicates.to_csv("eg_drugs_barcode_duplicates.csv", index=False, encoding="utf-8-sig")
validation_review.to_csv("eg_drugs_validation_review.csv", index=False, encoding="utf-8-sig")

print("Saved:", OUTPUT)
print("Final shape:", work.shape)

Saved: eg_drugs_cleaned_product_matching.csv
Final shape: (26562, 59)


## 12. Final Matching Rules

### Strong product match
Use `product_match_key` when available. It combines normalized active ingredients, strength/concentration, dosage form, route, release type, and packaging/volume.

### Product-family match
Use `product_family_key` when packaging differs or when the application needs to show products from the same family.

### Fuzzy matching
If a product has no exact key or name spelling is inconsistent, fuzzy matching can be used only to generate **candidate matches**. It should not automatically merge pharmaceutical records.

### Price comparison
Only compare prices after the products pass the appropriate matching rule. Then filter by valid numeric price and explicitly known positive availability.
